<a href="https://colab.research.google.com/github/andilMc/gemmafro-e2b/blob/main/notebooks/06_merge_adapter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 — Fusion de l'adaptateur LoRA en un modèle autonome

Fusionne l'adaptateur LoRA final (Phase 3) dans les poids du modèle de base et sauvegarde un modèle Gemma complet et autonome sur Drive. Objectif : ne plus jamais retélécharger le modèle de base depuis Hugging Face dans 04/05 — juste recharger ce dossier, sans PeftModel ni déballage de `Gemma4ClippableLinear`.


In [1]:
# Installe les dépendances.
!pip install -q -U transformers accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 53.3 MB/s eta 0:00:00


In [2]:
# Monte Drive et localise l'adaptateur LoRA final produit en Phase 3.
import os
from google.colab import drive

drive.mount('/content/drive')  # demande l'autorisation d'accès au Drive

PROJECT_DIR = '/content/drive/MyDrive/gemmafro-e2b'  # racine du projet sur Drive
ADAPTER_DIR = f'{PROJECT_DIR}/checkpoints/gemma-4-e2b-lora-final'  # adaptateur LoRA final de la Phase 3
MERGED_DIR = f'{PROJECT_DIR}/checkpoints/gemma-4-e2b-merged'  # modèle autonome, cible de ce notebook
assert os.path.isdir(ADAPTER_DIR), "Adaptateur introuvable — exécuter 03_finetune.ipynb jusqu'au bout d'abord."

Mounted at /content/drive


In [3]:
# Se connecte à Hugging Face — dernière fois nécessaire pour ce modèle : après la fusion, MERGED_DIR
# se suffit à lui-même et ce login ne sera plus requis pour le recharger.
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))  # authentifie la session avec le token HF_TOKEN

In [4]:
# Charge le modèle de base en bf16, PAS en 4-bit : fusionner l'adaptateur sur des poids déjà quantifiés
# perdrait en précision. La quantification 4-bit ne sera réappliquée qu'au chargement, dans 04/05.
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_NAME = "google/gemma-4-E2B-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)  # tokenizer Gemma
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # pas de token de padding défini : on réutilise le token de fin

gc.collect()  # libère la mémoire (Python, puis cache GPU)
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(  # poids complets, non quantifiés
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},  # tout sur le GPU 0, sans offload CPU
)
base_model.config.pad_token_id = tokenizer.pad_token_id  # aligne le modèle sur le token de padding du tokenizer

free_mem, total_mem = torch.cuda.mem_get_info()
print(f"Mémoire GPU libre après chargement du modèle bf16 : {free_mem / 1e9:.2f} / {total_mem / 1e9:.2f} Go")

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Mémoire GPU libre après chargement du modèle bf16 : 74.26 / 85.09 Go


In [5]:
# Déballe Gemma4ClippableLinear comme en Phases 3/4/5 : l'adaptateur a été entraîné sur cette structure,
# PeftModel doit la retrouver à l'identique pour rattacher les poids LoRA aux bons endroits.
def unwrap_clippable_linears(model):
    """Remplace Gemma4ClippableLinear par son Linear interne : PEFT ne sait attacher de LoRA que sur
    les types de couches qu'il reconnaît."""
    count = 0
    for module in model.modules():
        for child_name, child in list(module.named_children()):
            if child.__class__.__name__ == "Gemma4ClippableLinear":
                setattr(module, child_name, child.linear)
                count += 1
    print(f'{count} couches Gemma4ClippableLinear déballées')
    return model

base_model = unwrap_clippable_linears(base_model)  # déballage avant de charger l'adaptateur

232 couches Gemma4ClippableLinear déballées


In [8]:
import gc
import torch
!pip install --upgrade torchao
# Rattache l'adaptateur LoRA puis fusionne ses poids dans le modèle de base.
# merge_and_unload() renvoie un modèle Hugging Face standard : plus de wrapper PEFT, plus d'adaptateur
# séparé — juste un modèle complet dont les poids intègrent déjà le fine-tuning.
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)  # rattache les poids LoRA au modèle de base
merged_model = peft_model.merge_and_unload()  # additionne les poids LoRA dans le modèle de base
merged_model.eval()  # mode inférence (désactive le dropout)

gc.collect()  # libère la mémoire (Python, puis cache GPU)
torch.cuda.empty_cache()
print(f"Mémoire GPU allouée après fusion : {torch.cuda.memory_allocated() / 1e9:.2f} Go")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 70.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Mémoire GPU allouée après fusion : 10.22 Go


In [9]:
# Sauvegarde le modèle fusionné et le tokenizer sur Drive : ce dossier se suffit à lui-même, plus besoin
# de repasser par Hugging Face ni par l'adaptateur séparé pour en refaire une inférence, où que ce soit.
os.makedirs(MERGED_DIR, exist_ok=True)  # crée le dossier s'il n'existe pas
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)  # poids au format safetensors
tokenizer.save_pretrained(MERGED_DIR)

total_size = sum(
    os.path.getsize(os.path.join(MERGED_DIR, f))
    for f in os.listdir(MERGED_DIR)
    if os.path.isfile(os.path.join(MERGED_DIR, f))
)  # taille réelle sur disque, pour vérifier la place disponible sur Drive
print(f'Modèle fusionné sauvegardé dans {MERGED_DIR} ({total_size / 1e9:.2f} Go)')
print(os.listdir(MERGED_DIR))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle fusionné sauvegardé dans /content/drive/MyDrive/gemmafro-e2b/checkpoints/gemma-4-e2b-merged (10.24 Go)
['config.json', 'generation_config.json', 'model.safetensors', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


## Vérification : recharger le modèle fusionné, sans PeftModel ni Hugging Face

Simule exactement ce que feront 04/05 après cette fusion : chargement direct depuis Drive, avec quantification 4-bit reconstituée localement au chargement (rapide, sans téléchargement).

In [10]:
# Libère le modèle bf16 (non quantifié, volumineux) avant de recharger la version 4-bit à tester.
del base_model, peft_model, merged_model
gc.collect()
torch.cuda.empty_cache()

In [11]:
# Recharge depuis Drive uniquement — aucun accès à Hugging Face à partir d'ici.
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(  # même quantification 4-bit que dans 04/05
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

test_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
if test_tokenizer.pad_token is None:
    test_tokenizer.pad_token = test_tokenizer.eos_token
test_tokenizer.padding_side = 'left'  # génération par batch : les nouveaux tokens partent de la même position

test_model = AutoModelForCausalLM.from_pretrained(  # chargement direct, plus de PeftModel
    MERGED_DIR, quantization_config=bnb_config, device_map={"": 0},
)
test_model.config.pad_token_id = test_tokenizer.pad_token_id
test_model.eval()
print(f"Mémoire GPU allouée : {torch.cuda.memory_allocated() / 1e9:.2f} Go")

Loading weights:   0%|          | 0/791 [00:00<?, ?it/s]

[transformers] Gemma4ForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/gemmafro-e2b/checkpoints/gemma-4-e2b-merged
Key                                                                       | Status     | 
--------------------------------------------------------------------------+------------+-
model.audio_tower.layers.{0...11}.lconv1d.linear_start.weight             | UNEXPECTED | 
model.vision_tower.encoder.layers.{0...15}.self_attn.o_proj.weight        | UNEXPECTED | 
model.audio_tower.layers.{0...11}.feed_forward1.ffw_layer_2.weight        | UNEXPECTED | 
model.audio_tower.layers.{0...11}.self_attn.post.weight                   | UNEXPECTED | 
model.audio_tower.layers.{0...11}.feed_forward2.ffw_layer_2.weight        | UNEXPECTED | 
model.vision_tower.encoder.layers.{0...14}.self_attn.v_proj.weight        | UNEXPECTED | 
model.vision_tower.encoder.layers.{0...15}.mlp.gate_proj.weight           | UNEXPECTED | 
model.vision_tower.encoder.layers.{0...15}.mlp.down_proj.we

Mémoire GPU allouée : 17.63 Go


In [12]:
# Test de génération rapide pour confirmer que le modèle fusionné répond bien comme le modèle fine-tuné.
messages = [{"role": "user", "content": "Réponds à la question de santé suivante en Swahili, de façon claire "
                                           "et médicalement fiable.\n\nQuestion : Malaria ni nini?"}]
prompt = test_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = test_tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(test_model.device)

with torch.no_grad():
    out = test_model.generate(**inputs, max_new_tokens=150, do_sample=False, pad_token_id=test_tokenizer.pad_token_id)

new_tokens = out[:, inputs['input_ids'].shape[1]:]  # retire le prompt : ne garde que les tokens générés
print(test_tokenizer.decode(new_tokens[0], skip_special_tokens=True))

Malaria ni ugonjwa mbaya ya usingo ambayo husababishwa na aina ndogo ya omughonjwa kutoka kwa nyani.


---
**Modèle autonome prêt** : `checkpoints/gemma-4-e2b-merged` sur Drive. Dans `04_evaluate.ipynb` et `05_generate_submission.ipynb`, remplacez le bloc de chargement (base 4-bit + `PeftModel` + déballage) par un chargement direct de ce dossier — plus besoin de login Hugging Face ni de l'adaptateur séparé.